In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, ChiSqSelector, Normalizer, StringIndexer
from pyspark.ml.classification import LinearSVC, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [2]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("Text Classification with SVM") \
    .getOrCreate()

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/lib/spark/jars/slf4j-log4j12-1.7.30.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/lib/hadoop/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.slf4j.impl.Log4jLoggerFactory]
24/05/26 17:46:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/05/26 17:46:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/05/26 17:46:09 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/05/26 17:46:09 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
24/05/26 17:46:09 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
24/05/26 17:46:0

In [3]:
# Load the data
data_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json"
data = spark.read.json(data_path)
data = data.select("reviewText", "category").na.drop()

In [4]:
# Define the stages of the pipeline
indexer = StringIndexer(inputCol="category", outputCol="label")

pattern = r'\s+|\d+|[()\[\]{}.,;:!?\+\-=_\"\'`~#@&*%€$§\\\/]+'
regex_tokenizer = RegexTokenizer(inputCol="reviewText", outputCol="words", pattern=pattern, gaps=True, minTokenLength=2)

stop_words_remover = StopWordsRemover(inputCol="words", outputCol="filtered")

vectorizer = CountVectorizer(inputCol="filtered", outputCol="vectorized_Features")
idf = IDF(inputCol="vectorized_Features", outputCol="features")

chi_selector = ChiSqSelector(numTopFeatures=2000, featuresCol="features", outputCol="selectedFeatures", labelCol="label")

normalizer = Normalizer(inputCol="selectedFeatures", outputCol="normFeatures", p=2.0)

In [5]:
# Part 2: Building the pipeline and processing the data
pipeline = Pipeline(stages=[indexer, regex_tokenizer, stop_words_remover, vectorizer, idf, chi_selector])
fitted_pipeline = pipeline.fit(data)
transformed_data = fitted_pipeline.transform(data)

24/05/26 17:47:10 WARN DAGScheduler: Broadcasting large task binary with size 1061.4 KiB
24/05/26 17:47:19 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
24/05/26 17:47:19 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
24/05/26 17:47:26 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB


In [6]:
# Extracting and saving selected terms
chi_selector_model = fitted_pipeline.stages[-1]
selected_indices = chi_selector_model.selectedFeatures
vocabulary = fitted_pipeline.stages[3].vocabulary
selected_words = [vocabulary[i] for i in selected_indices]
sorted_words = sorted(selected_words)
words_line = " ".join(sorted_words)

with open("output_ds.txt", "w") as file:
    file.write(words_line)

In [7]:
# Part 3: Extending the pipeline for text classification with SVM
svm = LinearSVC(maxIter=10, regParam=0.1)
ovr = OneVsRest(classifier=svm, featuresCol="normFeatures", labelCol="label")

In [8]:
# Adding normalizer and SVM classifier to the pipeline
full_pipeline = Pipeline(stages=[indexer, regex_tokenizer, stop_words_remover, vectorizer, idf, chi_selector, normalizer, ovr])

In [9]:
# Split the data into training and test sets
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

In [12]:
paramGrid = (ParamGridBuilder()
    .addGrid(svm.regParam, [0.01, 0.1, 1.0])
    .addGrid(svm.standardization, [True, False])
    .addGrid(svm.maxIter, [10, 100])
    .build())

In [13]:
# Define the evaluator
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

In [14]:
# Cross-validation
crossval = CrossValidator(estimator=full_pipeline,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

In [ ]:
# Train the model using cross-validation
cv_model = crossval.fit(train_data)

24/05/26 17:49:25 WARN DAGScheduler: Broadcasting large task binary with size 1896.0 KiB
24/05/26 17:49:25 WARN DAGScheduler: Broadcasting large task binary with size 1898.1 KiB
24/05/26 17:49:29 WARN DAGScheduler: Broadcasting large task binary with size 1901.1 KiB
24/05/26 17:49:46 WARN DAGScheduler: Broadcasting large task binary with size 1915.3 KiB
24/05/26 17:49:50 WARN DAGScheduler: Broadcasting large task binary with size 1916.4 KiB
24/05/26 17:49:53 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/05/26 17:49:53 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
24/05/26 17:49:54 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
24/05/26 17:49:54 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
24/05/26 17:49:54 WARN DAGScheduler: Broadcasting large task binary with size 1916.4 KiB
24/05/26 

In [ ]:
# Make predictions on the test data
predictions = cv_model.transform(test_data)

In [ ]:
# Evaluate the model
f1_score = evaluator.evaluate(predictions)
print(f"F1 Score: {f1_score}")

In [ ]:
# Save the model if needed
cv_model.save("text_classification_svm_model")

In [ ]:
# Stop Spark session
spark.stop()
